In [1]:
# 키워드 기반 데이터 필터, 데이터프레임 생성, 타입 정리
from __future__ import annotations
from typing import Dict, Any, List
from collections.abc import Sequence, Iterable
import pandas as pd


COLUMN_MAP={
    "제목": "title",
    "장소": "location",
    "주최": "organizer",
    "시작 일시": "start_date",
    "종료 일시": "end_date",
    "주제 요약": "summary",
    "행사 성격": "event_type",
    "주요 키워드": "keywords",
    "출처": "source",
    "등록 링크": "registration_link",
    "상세 정보 링크": "more_info_link",
    "유료 여부": "is_free",
}

In [2]:
from typing import Any

import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from psycopg2.extras import RealDictCursor
from functools import lru_cache


load_dotenv()


def get_db_connection():
    try:
        conn = psycopg2.connect(
            host=os.getenv("DB_HOST"),
            port=os.getenv("DB_PORT"),
            database=os.getenv("DB_NAME"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
        )
        print("[Debug] DB 연결 성공")
        return conn

    except Exception as e:
        print(f"[Error] DB 연결 실패: {e}")
        raise


class EventRepository:
    @lru_cache(maxsize=1)
    def load_events_dataframe(self) -> pd.DataFrame:
        query = """
            SELECT event_id, payload::jsonb -> 'source_row' AS source_row
            FROM public.events
            WHERE jsonb_typeof(payload::jsonb -> 'source_row') = 'object'
        """

        conn = get_db_connection()

        try:
            with conn.cursor(cursor_factory=RealDictCursor) as cur:
                cur.execute(query)
                rows: list[dict[str, Any]] = cur.fetchall()
        finally:
                conn.close()

        records: list[dict[str, Any]] = []

        for row in rows:
            source_row = dict(row["source_row"])
            source_row["event_id"] = str(row["event_id"])
            records.append(source_row)

        event_df = pd.DataFrame(records)
        print(f"[debug] DB조회 : {event_df.head()}")
        return event_df

    def clear_cache(self):
        self.load_events_dataframe.cache_clear()

In [3]:
repository = EventRepository()
event_df = repository.load_events_dataframe()

print(event_df.head())

[Debug] DB 연결 성공
[debug] DB조회 :    Index                          장소  \
0      1               Seattle, WA |   
1      2  한양대학교 국제관 6층 602호 (디지털강의실)   
2      3                   서울 대한전기협회   
3      4                         미기재   
4      5                         미기재   

                                                  제목  \
0                              U.S. Women in Nuclear   
1                 2026년 iTRS 몬테칼로 이론 및 실무교육(MCNP) 안내   
2                                          일반기계 공인검사   
3  Webinar: NEA–AFCONE Engagement Series with Afr...   
4                              가공배전 교육 (30일/240H) 94   

                           주최                               출처  \
0                   한국원자력산업협회                KAIF 원자력계 일정 AJAX   
1                     한국원자력학회         KNS news 게시판 제목 순회·본문 확인   
2                대한전기협회 KEPIC                      KEPIC 교육 일정   
3  OECD Nuclear Energy Agency  OECD NEA generated.Event search   
4              대한전기협회 전력기술교육원                   KEA 전력기술교육원 

In [4]:
def get_filter_options_data(df: pd.DataFrame) -> Dict[str, List[str]]:
    """DataFrame에서 필터 드롭다운을 위한 유니크 값들 추출"""
    if df.empty:
        return {"organizers": [], "event_types": [], "keywords": [], "locations": []}

    organizers = df["주최"].dropna().unique().tolist() if "주최" in df.columns else []
    event_types = df["행사 성격"].dropna().unique().tolist() if "행사 성격" in df.columns else []
    locations = df["장소"].dropna().unique().tolist() if "장소" in df.columns else []
    
    keywords_set = set()
    if "주요 키워드" in df.columns:
        for kw_string in df["주요 키워드"].dropna():
            keywords_set.update([k.strip() for k in str(kw_string).split(",") if k.strip()])

    result = {
        "organizers": sorted(organizers),
        "event_types": sorted(event_types),
        "keywords": sorted(list(keywords_set)),
        "locations": sorted(locations),
    }
    print(f"\n\n[DEBUG] 필터 옵션 드롭다운값 : {result} \n")
    return result

In [5]:
get_filter_options_data(event_df)



[DEBUG] 필터 옵션 드롭다운값 : {'organizers': ['IAEA', 'IEA; Government of Austria', 'IEA; International Emissions Trading Association (IETA); EPRI', 'IEA; Republic of Kenya; African Union Commission; African Development Bank; U.S. Department of Energy; Norway', 'IEA; South Africa Department of Electricity and Energy', 'IEEE PES', 'IEEE PES (재정 또는 기술 후원)', 'International Energy Agency (IEA)', 'OECD Nuclear Energy Agency', '대한전기협회', '대한전기협회 KEPIC', '대한전기협회 전력기술교육원', '한국원자력산업협회', '한국원자력학회'], 'event_types': ['Conference', 'KEC 교육', 'KEPIC 교육', 'Report launch', 'Training', 'Workshop', '공개 행사', '공지에서 확인된 행사', '교육·튜토리얼', '국제학술대회', '기술회의', '세미나·행사', '워크숍', '웨비나', '위원회 회의', '전력기술교육원 교육', '컨퍼런스', '학술대회·행사', '행사·교육'], 'keywords': ['Africa', 'ETS', 'IAEA', 'KAIF', 'KEC', 'KEPIC', 'KNS', 'OECD', 'OECD NEA', 'carbon markets', 'clean cooking', 'competitiveness', 'emissions trading', 'energy efficiency', 'energy innovation', 'energy security', 'oil market', 'policy', 'report launch', 'training', '계통 해석·제어',

{'organizers': ['IAEA',
  'IEA; Government of Austria',
  'IEA; International Emissions Trading Association (IETA); EPRI',
  'IEA; Republic of Kenya; African Union Commission; African Development Bank; U.S. Department of Energy; Norway',
  'IEA; South Africa Department of Electricity and Energy',
  'IEEE PES',
  'IEEE PES (재정 또는 기술 후원)',
  'International Energy Agency (IEA)',
  'OECD Nuclear Energy Agency',
  '대한전기협회',
  '대한전기협회 KEPIC',
  '대한전기협회 전력기술교육원',
  '한국원자력산업협회',
  '한국원자력학회'],
 'event_types': ['Conference',
  'KEC 교육',
  'KEPIC 교육',
  'Report launch',
  'Training',
  'Workshop',
  '공개 행사',
  '공지에서 확인된 행사',
  '교육·튜토리얼',
  '국제학술대회',
  '기술회의',
  '세미나·행사',
  '워크숍',
  '웨비나',
  '위원회 회의',
  '전력기술교육원 교육',
  '컨퍼런스',
  '학술대회·행사',
  '행사·교육'],
 'keywords': ['Africa',
  'ETS',
  'IAEA',
  'KAIF',
  'KEC',
  'KEPIC',
  'KNS',
  'OECD',
  'OECD NEA',
  'carbon markets',
  'clean cooking',
  'competitiveness',
  'emissions trading',
  'energy efficiency',
  'energy innovation',
  'energy secur